# Autodiff Policy Optimization — Optimizer Swap

**Context:** Track B (`sparse-policy.ipynb`) fits the polynomial `u = Θ(x) @ ξ` via
STLSQ, which minimises open-loop imitation error on the training dataset. This notebook
keeps the **same polynomial structure** but replaces the STLSQ objective with gradient
descent through simulated rollouts — an *optimizer swap*, not a model swap.

| | STLSQ (Track B) | Autodiff (this notebook) |
|---|---|---|
| **Objective** | `‖Θ(X) @ ξ − U‖²`  (imitation) | `Σ_t cost(x_t)` along rollout |
| **Information** | expert actions at training states | closed-loop trajectory cost |
| **Compounding error** | ignored at fit time | directly penalized |

**Setup:**
1. Fit augmented SINDy (≡ Track B Part 3) — provides the warm-start for `ξ`
2. Learn a residual dynamics model `x_{t+1} = x_t + f_θ(x_t, u_t)` from baseline data
3. Optimize `ξ` by backpropagating through multi-step rollouts in the learned model
4. Evaluate in real MuJoCo and diagnose model overfitting

In [ ]:
import sys
import pathlib
from itertools import combinations_with_replacement

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import pysindy as ps
import torch
import torch.nn as nn
from stable_baselines3 import PPO

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from plotting_utils import plot_eval_bars

# ── Config ─────────────────────────────────────────────────────────────────────
ENV_ID       = "InvertedDoublePendulum-v5"
SINDY_DEG    = 2          # must match sparse-policy.ipynb selection
SINDY_THRS   = 0.01
DATA_PATH    = pathlib.Path("../data/baseline/trajectories_baseline.npz")
CHECKPOINT  = "../data/baseline/checkpoints/best_model.zip"
STATE_LABELS = ["x", "θ₁", "θ₂", "ẋ", "θ̇₁", "θ̇₂"]
NOISE_LEVELS = [0.01, 0.05, 0.1, 0.15, 0.2, 0.3]
AUG_NOISE    = 0.05
AUG_REPEATS  = 3
N_EVAL       = 20

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_env      = gym.make(ENV_ID)
MAX_STEPS = _env.spec.max_episode_steps
DT        = _env.unwrapped.dt
EVAL_NOISE = getattr(_env.unwrapped, "reset_noise_scale",
                     getattr(_env.unwrapped, "_reset_noise_scale", 0.1))
_env.close()
print(f"MAX_STEPS={MAX_STEPS}  DT={DT}s  EVAL_NOISE={EVAL_NOISE}  device={DEVICE}")


# ── Shared helpers ─────────────────────────────────────────────────────────────
def obs_to_state6(obs):
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]),
        np.arctan2(obs[2], obs[4]),
        obs[5], obs[6], obs[7],
    ], dtype=np.float64)


def state6_to_obs9(s):
    x, th1, th2, xdot, th1dot, th2dot = s
    return np.array([
        x, np.sin(th1), np.sin(th2), np.cos(th1), np.cos(th2),
        xdot, th1dot, th2dot, 0.0,
    ], dtype=np.float64)


def make_sindy_policy(coeff, library):
    def policy(state6):
        Th = library.transform(np.asarray(state6, dtype=np.float64).reshape(1, -1))
        u  = float(np.asarray(Th @ coeff.T).flatten()[0])
        return np.array([np.clip(u, -1.0, 1.0)], dtype=np.float32)
    return policy


def evaluate_sindy(policy_fn, noise_std=0.0, n_episodes=10, seed=0):
    ep_lens, ep_rews = [], []
    for ep in range(n_episodes):
        env = gym.make(ENV_ID, reset_noise_scale=noise_std)
        obs, _ = env.reset(seed=seed + ep)
        ep_r = ep_l = 0
        done = False
        while not done:
            obs, r, term, trunc, _ = env.step(policy_fn(obs_to_state6(obs)))
            ep_r += r; ep_l += 1
            done = term or trunc
        ep_lens.append(ep_l); ep_rews.append(ep_r)
        env.close()
    return np.array(ep_lens), np.array(ep_rews)

## Warm-Start: Augmented SINDy

Replicate Track B Part 3 in a single cell to obtain `coeff_a` — the augmented STLSQ
solution that serves as the warm-start for `ξ`. Starting from the best imitation-learning
solution isolates the effect of switching objectives from the effect of initialization.

In [ ]:
data = np.load(DATA_PATH)
X = data["X"].astype(np.float64)
U = data["U"].astype(np.float64).reshape(-1, 1)

# Augment: query expert NN at perturbed states
nn_model = PPO.load(CHECKPOINT)
dev_nn   = nn_model.policy.device
X_parts, U_parts = [X], [U]

for rep in range(AUG_REPEATS):
    X_pert = X + np.random.normal(0.0, AUG_NOISE, X.shape)
    obs9   = np.array([state6_to_obs9(s) for s in X_pert], dtype=np.float32)
    with torch.no_grad():
        u_pert = (
            nn_model.policy.get_distribution(torch.as_tensor(obs9).to(dev_nn))
            .get_actions(deterministic=True)
            .cpu().numpy().reshape(-1, 1).astype(np.float64)
        )
    X_parts.append(X_pert); U_parts.append(u_pert)

X_aug = np.vstack(X_parts)
U_aug = np.vstack(U_parts)

# Fit augmented SINDy — warm-start for autodiff
library_aug = ps.PolynomialLibrary(degree=SINDY_DEG)
library_aug.fit(X_aug)
Theta_aug     = library_aug.transform(X_aug)
feature_names = library_aug.get_feature_names_out(STATE_LABELS)
opt_aug       = ps.STLSQ(threshold=SINDY_THRS)
opt_aug.fit(Theta_aug, U_aug)
coeff_a       = opt_aug.coef_  # (1, n_features)

nz = int(np.sum(np.abs(coeff_a) > 1e-8))
u_pred = np.asarray((Theta_aug @ coeff_a.T).flatten())
rmse   = float(np.sqrt(np.mean((U_aug.flatten() - u_pred) ** 2)))
r2     = 1.0 - np.sum((U_aug.flatten() - u_pred)**2) / np.sum((U_aug.flatten() - U_aug.mean())**2)
print(f"Augmented SINDy  nonzero={nz}/{Theta_aug.shape[1]}  RMSE={rmse:.4f}  R²={r2:.4f}")

sindy_policy_aug = make_sindy_policy(coeff_a, library_aug)

---

## Part 1 — Differentiable World Model

MuJoCo is not differentiable, so we learn a surrogate:

$$\hat{\mathbf{x}}_{t+1} = \mathbf{x}_t + f_\theta(\mathbf{x}_t,\, u_t)$$

The residual form (`x + MLP(x,u)`) keeps the identity as the default prediction, which
accelerates learning and improves gradient flow. The model is trained on the baseline
`(X, U, X_next)` transitions, then **frozen** — only `ξ` is optimized in Part 2.

In [ ]:
class ResidualDynamics(nn.Module):
    def __init__(self, state_dim=6, action_dim=1, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),                 nn.Tanh(),
            nn.Linear(hidden, state_dim),
        )
    def forward(self, x, u):
        return x + self.net(torch.cat([x, u], dim=-1))

X_t  = torch.tensor(data["X"],      dtype=torch.float32, device=DEVICE)
U_t  = torch.tensor(data["U"].reshape(-1, 1), dtype=torch.float32, device=DEVICE)
Xn_t = torch.tensor(data["X_next"], dtype=torch.float32, device=DEVICE)

dyn     = ResidualDynamics().to(DEVICE)
opt_dyn = torch.optim.Adam(dyn.parameters(), lr=3e-3)

DYN_EPOCHS, DYN_BATCH = 200, 512
print("Training dynamics model ...")
for epoch in range(DYN_EPOCHS):
    perm, losses = torch.randperm(len(X_t), device=DEVICE), []
    for i in range(0, len(X_t), DYN_BATCH):
        bi   = perm[i:i+DYN_BATCH]
        loss = (dyn(X_t[bi], U_t[bi]) - Xn_t[bi]).pow(2).mean()
        opt_dyn.zero_grad(); loss.backward(); opt_dyn.step()
        losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"  epoch {epoch+1:3d}/{DYN_EPOCHS}  MSE={np.mean(losses):.6f}")

dyn.eval()
for p in dyn.parameters():
    p.requires_grad_(False)  # freeze — only ξ gets gradients

with torch.no_grad():
    rmse_dyn = (dyn(X_t, U_t) - Xn_t).pow(2).mean().sqrt().item()
print(f"\nDynamics RMSE on training data: {rmse_dyn:.5f}")
print("(Lower is better; RMSE < 0.01 suggests accurate one-step predictions)")

---

## Part 2 — Optimizer Swap: ξ via Gradient Descent

With the dynamics model frozen, `ξ` is the only free parameter. At each gradient step:

1. Sample a batch of initial states `x_0` from the training distribution
2. Roll out `H` steps: `x_{t+1} = dyn(x_t, clip(Θ(x_t) @ ξ, -1, 1))`
3. Accumulate cost: `Σ_t (θ₁² + θ₂² + 0.1·x²)` — poles upright, cart centered
4. Backpropagate through the entire rollout into `ξ`

The gradient chain is: `ξ → u_t → x_{t+1} = dyn(x_t, u_t) → x_{t+2} → ... → Σ cost`

Gradient clipping prevents explosion across the `H`-step rollout.

In [ ]:
def poly_features_torch(X: torch.Tensor, degree: int = 2) -> torch.Tensor:
    """Polynomial features matching PySINDy PolynomialLibrary — any degree."""
    n, d = X.shape
    feats = [torch.ones(n, 1, device=X.device, dtype=X.dtype)]
    for deg in range(1, degree + 1):
        for idx in combinations_with_replacement(range(d), deg):
            feat = X[:, idx[0]]
            for i in idx[1:]:
                feat = feat * X[:, i]
            feats.append(feat.unsqueeze(1))
    return torch.cat(feats, dim=1)


# Warm-start from augmented SINDy
xi     = nn.Parameter(torch.tensor(coeff_a, dtype=torch.float32, device=DEVICE))
opt_xi = torch.optim.Adam([xi], lr=5e-3)

ROLLOUT_LEN = 20
N_OPT_STEPS = 500
BATCH_OPT   = 128

imagined_costs = []
print("Optimizing ξ via autodiff through rollouts ...")
for step in range(N_OPT_STEPS):
    idx = torch.randint(0, len(X_t), (BATCH_OPT,), device=DEVICE)
    x   = X_t[idx].clone()

    total_cost = torch.zeros(1, device=DEVICE)
    for _ in range(ROLLOUT_LEN):
        Th         = poly_features_torch(x, degree=SINDY_DEG)
        u          = torch.clamp(Th @ xi.T, -1.0, 1.0)
        total_cost = total_cost + (x[:, 1].pow(2) + x[:, 2].pow(2) + 0.1 * x[:, 0].pow(2)).mean()
        x          = dyn(x, u)

    loss = total_cost / ROLLOUT_LEN
    opt_xi.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_([xi], max_norm=1.0)
    opt_xi.step()
    imagined_costs.append(loss.item())

    if (step + 1) % 100 == 0:
        print(f"  step {step+1:4d}/{N_OPT_STEPS}  imagined_cost={loss.item():.4f}")

coeff_ad = xi.detach().cpu().numpy()

# Learning curve
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(imagined_costs, lw=1.2, color="mediumorchid")
ax.set_xlabel("Gradient step"); ax.set_ylabel("Imagined rollout cost")
ax.set_title("Autodiff optimization — convergence in learned dynamics")
plt.tight_layout(); plt.show()

print("\nCoefficient shift (|c| > 0.01):   [STLSQ-aug] → [autodiff]")
for name, ca, cd in zip(feature_names, coeff_a[0], coeff_ad[0]):
    if abs(ca) > 0.01 or abs(cd) > 0.01:
        print(f"  {name:20s}  {ca:+.4f}  →  {cd:+.4f}")

---

## Part 3 — Evaluation & Overfitting Diagnostic

The key diagnostic: compare **imagined cost** (in the learned model) vs **real episode
length** (in MuJoCo). A large gap indicates the autodiff policy overfitted to errors in
the dynamics model — it found coefficients that look good in imagination but fail in
reality.

The perturbation sweep extends this: if the autodiff curve lies *below* the augmented
SINDy curve at high noise, the dynamics model is the bottleneck, not the optimization.

In [ ]:
library_ad   = ps.PolynomialLibrary(degree=SINDY_DEG)
library_ad.fit(X)
sindy_policy_ad = make_sindy_policy(coeff_ad, library_ad)

# ── Overfitting diagnostic ─────────────────────────────────────────────────────
final_imagined_cost = float(np.mean(imagined_costs[-50:]))  # last 50 steps
lens_ad_nominal, _ = evaluate_sindy(sindy_policy_ad, noise_std=EVAL_NOISE, n_episodes=N_EVAL)
mean_real_len = float(np.mean(lens_ad_nominal))

print("Overfitting diagnostic")
print(f"  Final imagined rollout cost : {final_imagined_cost:.4f}")
print(f"  Mean MuJoCo episode length  : {mean_real_len:.1f} / {MAX_STEPS} steps")
print(f"  Completion rate             : {100 * np.mean(lens_ad_nominal == MAX_STEPS):.0f}%")
print()
print("Interpretation:")
print("  Low imagined cost + high real length  → optimizer swap is working")
print("  Low imagined cost + low  real length  → policy overfit to dynamics model errors")

# ── Perturbation sweep ─────────────────────────────────────────────────────────
print("\nPerturbation sweep — augmented SINDy ...")
sweep_aug = []
for σ in NOISE_LEVELS:
    lens, _ = evaluate_sindy(sindy_policy_aug, noise_std=σ, n_episodes=10)
    sweep_aug.append(lens.mean())
    print(f"  noise_std={σ:.2f}  mean_ep_len={lens.mean():.0f}")

print("\nPerturbation sweep — autodiff ξ ...")
sweep_ad = []
for σ in NOISE_LEVELS:
    lens, _ = evaluate_sindy(sindy_policy_ad, noise_std=σ, n_episodes=10)
    sweep_ad.append(lens.mean())
    print(f"  noise_std={σ:.2f}  mean_ep_len={lens.mean():.0f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(NOISE_LEVELS, sweep_aug, "s-", color="darkorange",   lw=1.8, ms=6, label="SINDy (augmented STLSQ)")
ax.plot(NOISE_LEVELS, sweep_ad,  "^-", color="mediumorchid", lw=1.8, ms=6, label="SINDy (autodiff ξ)")
ax.axhline(MAX_STEPS, color="green", ls="--", lw=1.5, label=f"max ({MAX_STEPS} steps)")
ax.set_xlabel("Reset noise std (rad / m)")
ax.set_ylabel("Mean episode length (steps)")
ax.set_title("Robustness: augmented STLSQ vs autodiff ξ\n(gap at high noise = dynamics-model overfitting)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# ── Bar chart ──────────────────────────────────────────────────────────────────
lens_aug, rews_aug = evaluate_sindy(sindy_policy_aug, noise_std=EVAL_NOISE, n_episodes=N_EVAL)
lens_ad,  rews_ad  = evaluate_sindy(sindy_policy_ad,  noise_std=EVAL_NOISE, n_episodes=N_EVAL)

print(f"\nSINDy (augmented STLSQ) : {np.mean(lens_aug):.1f} steps  ({100*np.mean(lens_aug==MAX_STEPS):.0f}% complete)")
print(f"SINDy (autodiff ξ)      : {np.mean(lens_ad):.1f} steps  ({100*np.mean(lens_ad==MAX_STEPS):.0f}% complete)")

plot_eval_bars(
    {"SINDy (augmented STLSQ)": (lens_aug, rews_aug),
     "SINDy (autodiff ξ)":      (lens_ad,  rews_ad)},
    MAX_STEPS, DT,
    title="Autodiff optimizer swap — direct comparison",
)